# Phase 1 — Iris Recognition: Fine-tuning + Image Testing

Fine-tunes the iris embedding model used by `models/iris/inference.py` (ResNet18 + projection head, trained with an ArcFace head), evaluates it (Experiment 1), and runs an image-based testing section.

**⚠️ Dataset licensing note:** there is no fully open, unrestricted iris dataset. CASIA-Iris-Thousand is only available from the official source (http://biometrics.idealtest.org) after signing CASIA's license agreement; unofficial mirrors exist on Kaggle/Hugging Face but their redistribution rights are not guaranteed. **You are responsible for confirming you have the right to use whichever copy of the dataset you point this notebook at** — see `docs/DATASETS.md`. This notebook is dataset-agnostic: point `DATASET_ROOT` at any folder organized as `DATASET_ROOT/<identity>/*.jpg`.

## 1. Setup

In [ ]:
# --- Environment setup (no Google Drive mount required) ---
# Installs only what's missing on top of Colab's preinstalled torch/torchvision.
!pip install -q kagglehub h5py

import sys, os
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/content/repo"
# Phase 1 code currently lives on this branch (not yet merged to main/master) -
# update to the default branch once the phase-1 PR is merged.
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Some hosted GPU sessions intermittently hand out a driver/torch-build
# combination where CUDA reports available but no kernel image exists for
# the actual device ("CUDA error: no kernel image is available for
# execution on the device") - smoke-test with a real op now and fall back
# to CPU rather than crashing deep into training on a broken GPU.
if DEVICE == "cuda":
    try:
        (torch.zeros(1, device=DEVICE) + 1).cpu()
    except Exception as e:  # torch.AcceleratorError, RuntimeError, etc.
        print(f"CUDA smoke test failed ({e}); falling back to CPU.")
        DEVICE = "cpu"
print("Using device:", DEVICE)

## 2. Get the dataset
Two ways to populate `DATASET_ROOT`, pick one:

**(a) Kaggle** (only if you have confirmed you're allowed to use that copy):
```python
import kagglehub
DATASET_ROOT = kagglehub.dataset_download("sondosaabed/casia-iris-thousand")
```
**(b) Your own licensed copy**, uploaded to this Colab session or a Drive folder you mount yourself:
```python
DATASET_ROOT = "/content/your_iris_dataset"
```
Either way, the folder must contain one subfolder per identity, each holding that identity's eye images.

In [ ]:
DATASET_ROOT = None  # <-- set this to one of the options above before running

if DATASET_ROOT is None:
    raise ValueError(
        "Set DATASET_ROOT to a licensed iris dataset folder (see the markdown cell above) "
        "before continuing — no dataset is downloaded automatically because CASIA-Iris "
        "requires confirming license terms yourself."
    )

## 3. Load identities and images

In [ ]:
import os, cv2, numpy as np

identity_dirs = sorted([d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))])
identity_to_label = {name: i for i, name in enumerate(identity_dirs)}

raw_images, raw_labels = [], []
for name in identity_dirs:
    folder = os.path.join(DATASET_ROOT, name)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            img = cv2.imread(os.path.join(folder, fname))
            if img is None:
                continue
            raw_images.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            raw_labels.append(identity_to_label[name])

print(f"{len(raw_images)} images across {len(identity_dirs)} identities")

## 4. Preprocess (localize + normalize) every image
Uses `preprocessing/iris.py` (Hough circle localization + Daugman rubber-sheet normalization) so training matches production preprocessing exactly. Images where the pupil/iris boundary can't be localized are dropped.

In [ ]:
from preprocessing.iris import IrisPreprocessor

preprocessor = IrisPreprocessor()
strips, strip_labels = [], []
skipped = 0
for img, label in zip(raw_images, raw_labels):
    try:
        strips.append(preprocessor.preprocess(img))
        strip_labels.append(label)
    except ValueError:
        skipped += 1

strips = np.stack(strips)  # (N, H, W) grayscale
strip_labels = np.array(strip_labels)
print(f"Normalized {len(strips)} iris strips, skipped {skipped}")

## 5. Train / validation / test split (per identity, same rationale as the face notebook)

In [ ]:
from collections import defaultdict

rng = np.random.default_rng(42)
by_identity = defaultdict(list)
for idx, label in enumerate(strip_labels):
    by_identity[label].append(idx)

train_idx, val_idx, test_idx = [], [], []
for label, idxs in by_identity.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    n = len(idxs)
    n_train = max(1, int(n * 0.7))
    n_val = max(1, int(n * 0.15))
    train_idx.extend(idxs[:n_train])
    val_idx.extend(idxs[n_train:n_train + n_val])
    test_idx.extend(idxs[n_train + n_val:] if n - n_train - n_val > 0 else idxs[-1:])

print(f"train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## 6. Model: ResNet18 + projection head + ArcFace

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from models.iris.inference import IrisEmbeddingNet, IRIS_EMBEDDING_DIM
from models.common.arcface import ArcMarginProduct

num_classes = len(identity_dirs)
model = IrisEmbeddingNet().to(DEVICE)
arc_head = ArcMarginProduct(IRIS_EMBEDDING_DIM, num_classes).to(DEVICE)

# Freeze the early backbone layers, fine-tune layer4 + the projection head:
# ResNet18 is small enough that this still trains quickly on a single Colab GPU.
for name, param in model.backbone.named_parameters():
    param.requires_grad = name.startswith('7')  # '7' == layer4 index in the Sequential

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable, lr=1e-4)
criterion = nn.CrossEntropyLoss()

## 7. Training loop

In [ ]:
from torch.utils.data import DataLoader, Dataset

class IrisDataset(Dataset):
    def __init__(self, strips, labels):
        self.strips, self.labels = strips, labels

    def __len__(self):
        return len(self.strips)

    def __getitem__(self, i):
        strip = np.stack([self.strips[i]] * 3, axis=-1)  # replicate to 3 channels for the ImageNet backbone
        tensor = torch.from_numpy(strip).permute(2, 0, 1).float() / 255.0
        tensor = (tensor - 0.5) / 0.5
        return tensor, int(self.labels[i])

train_loader = DataLoader(IrisDataset(strips[train_idx], strip_labels[train_idx]), batch_size=32, shuffle=True)
val_loader = DataLoader(IrisDataset(strips[val_idx], strip_labels[val_idx]), batch_size=32)

NUM_EPOCHS = 15
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model(x)
        logits = arc_head(embeddings, y)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += x.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            embeddings = model(x)
            logits = arc_head(embeddings, y)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += x.size(0)

    print(f"epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss/train_total:.4f} "
          f"train_acc={train_correct/train_total:.3f} val_acc={val_correct/max(val_total,1):.3f}")

## 8. Save the checkpoint

In [ ]:
from models.common.checkpoint_io import save_state_dict_as_h5

CHECKPOINT_PATH = f"{REPO_DIR}/models/iris/saved/iris_embedder.pt"
H5_PATH = f"{REPO_DIR}/models/iris/saved/iris_embedder.h5"
model.eval()
torch.save(model.state_dict(), CHECKPOINT_PATH)  # canonical, loaded by models/iris/inference.py
save_state_dict_as_h5(model.state_dict(), H5_PATH)  # interoperability export
print("Saved checkpoint to", CHECKPOINT_PATH, "and", H5_PATH)

## 9. Experiment 1 — recognition performance on the held-out test set

In [ ]:
from models.iris.inference import IrisEmbedder
from evaluation.experiments import run_modality_experiment
from evaluation.roc import plot_roc

fine_tuned_embedder = IrisEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load — check the path above."

test_embeddings = [fine_tuned_embedder.extract_embedding(strips[i]) for i in test_idx]
test_labels = [identity_dirs[strip_labels[i]] for i in test_idx]

report = run_modality_experiment(test_embeddings, test_labels, modality_name="iris")
print(f"Iris  |  Accuracy@EER-threshold: {report['accuracy_at_eer_threshold']:.3f}  EER: {report['eer']:.3f}  AUC: {report['auc']:.3f}")
plot_roc({"Iris (fine-tuned)": report["roc"]}, save_path=f"{REPO_DIR}/evaluation/results/iris_roc.png")

## 10. Test on images — genuine vs. impostor pair

In [ ]:
import matplotlib.pyplot as plt
from evaluation.metrics import cosine_similarity

test_idx_arr = np.array(test_idx)
test_label_names = np.array([identity_dirs[strip_labels[i]] for i in test_idx])

def pick_pair(same_identity: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_idx_arr), size=2, replace=False)
        if (test_label_names[i] == test_label_names[j]) == same_identity:
            return test_idx_arr[i], test_idx_arr[j]
    raise RuntimeError("Could not find a suitable pair in 200 tries")

genuine_a, genuine_b = pick_pair(same_identity=True)
impostor_a, impostor_b = pick_pair(same_identity=False)
threshold = report["eer_threshold"]

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for row, (a, b, expected) in enumerate([(genuine_a, genuine_b, "GENUINE"), (impostor_a, impostor_b, "IMPOSTOR")]):
    emb_a = fine_tuned_embedder.extract_embedding(strips[a])
    emb_b = fine_tuned_embedder.extract_embedding(strips[b])
    score = cosine_similarity(emb_a, emb_b)
    verdict = "MATCH" if score >= threshold else "NO MATCH"
    for col, idx in enumerate([a, b]):
        axes[row, col].imshow(strips[idx], cmap="gray")
        axes[row, col].set_title(identity_dirs[strip_labels[idx]], fontsize=9)
        axes[row, col].axis("off")
    fig.text(0.5, 1 - row * 0.5 - 0.03, f"{expected} pair — similarity={score:.3f} → {verdict} (threshold={threshold:.3f})",
             ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/evaluation/results/iris_pair_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Get the checkpoint back to your machine
Same two options as the face notebook (download + commit locally, or push directly from Colab using a `GITHUB_TOKEN` Colab secret).

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)
files.download(H5_PATH)